# Análisis de insights — Ecommerce

Carga del tablón analítico y métricas de embudo de eventos (base: **view** = 100 %).

In [1]:
from pathlib import Path

import pandas as pd

ruta_tablon = Path("..") / "datos" / "intermedios" / "tablon_analitico.pickle"
df = pd.read_pickle(ruta_tablon)
df.shape

(2074026, 24)

## Porcentajes de eventos (view = 100 %)

Se interpreta **view** como referencia. 

In [2]:
# Calcular el total de 'view' como base 100%
total_view = df.evento.value_counts().get('view', 0)

# Calcular el total de 'cart'
total_cart = df.evento.value_counts().get('cart', 0)

# Calcular el total de 'remove_from_cart'
total_remove = df.evento.value_counts().get('remove_from_cart', 0)

# Calcular el total de 'purchase'
total_purchase = df.evento.value_counts().get('purchase', 0)

# Porcentajes
porcentaje_cart_sobre_view = (total_cart / total_view) * 100 if total_view else 0
porcentaje_remove_sobre_cart = (total_remove / total_cart) * 100 if total_cart else 0
porcentaje_purchase_sobre_cart = (total_purchase / total_cart) * 100 if total_cart else 0

# Mostrar resultados
print(f"Cart/View: {porcentaje_cart_sobre_view:.2f}%")
print(f"Remove_from_cart/Cart: {porcentaje_remove_sobre_cart:.2f}%")
print(f"Purchase/Cart: {porcentaje_purchase_sobre_cart:.2f}%")



Cart/View: 59.75%
Remove_from_cart/Cart: 71.42%
Purchase/Cart: 22.20%


In [3]:
# Es mejor utilizar el método get para obtener el valor de un evento, porque si el evento no existe, devuelve 0.
df.evento.value_counts().get('view', 0)

np.int64(961558)

In [4]:
# si el evento no existe, devuelve un error. Evitamos este método.
df.evento.value_counts()['view']

np.int64(961558)

In [5]:
# Crear el DataFrame del funnel y modificamos todos los porcentajes sobre visualizaciones para que tenga forma de embudo.
funnel_data = {
    'evento': ['Visualizaciones', 'Añadir a carrito', 'Sacar de carrito', 'Comprar'],
    'porcentaje': [
        100,
        (total_cart / total_view) * 100 if total_view else 0,
        (total_remove / total_view) * 100 if total_view else 0,
        (total_purchase / total_view) * 100 if total_view else 0
    ]
}

df_funnel = pd.DataFrame(funnel_data)
print(df_funnel)

             evento  porcentaje
0   Visualizaciones  100.000000
1  Añadir a carrito   59.751674
2  Sacar de carrito   42.676261
3           Comprar   13.266386


In [11]:
import plotly.graph_objects as go

fig = go.Figure(go.Funnel(
    y=df_funnel['evento'],
    x=df_funnel['porcentaje'],
    textinfo='text',
    texttemplate='%{value:.1f}%',
))
fig.update_layout(title="Gráfico de Funnel de Conversión")
from IPython.display import display
display(fig)

In [12]:
df.groupby(['sesion','evento'])['producto'].count()

sesion                                evento
0000597b-de39-4a77-9fe5-02c8792ca14e  view      3
0000645a-8160-4a3d-91bf-154bff0a22e3  view      2
000090e1-da13-42b1-a31b-91a9ee5e6a88  view      1
0000b3cb-5422-4bf2-b8fe-5c1831d0dc1b  view      1
0000de26-bd58-42c9-9173-4763c76b398e  view      1
                                               ..
ffff6695-b64d-4a67-aa14-34b3b7f63c3f  view      2
ffff7d69-b706-4c64-9d6d-da57a04bc32b  view      1
ffff8044-2a22-4846-8a72-999e870abbe9  view      1
ffff91d4-7879-4a4b-8b26-c67915a27dc8  view      1
ffffbe0a-d2c2-47c7-afab-680bfdfda50d  view      1
Name: producto, Length: 581763, dtype: int64

In [14]:
#pasamos evento a columnas
sesiones = df.groupby(['sesion','evento'])['producto'].count().unstack().fillna(0)
sesiones

evento,cart,purchase,remove_from_cart,view
sesion,,,,
0000597b-de39-4a77-9fe5-02c8792ca14e,0.0,0.0,0.0,3.0
0000645a-8160-4a3d-91bf-154bff0a22e3,0.0,0.0,0.0,2.0
000090e1-da13-42b1-a31b-91a9ee5e6a88,0.0,0.0,0.0,1.0
0000b3cb-5422-4bf2-b8fe-5c1831d0dc1b,0.0,0.0,0.0,1.0
0000de26-bd58-42c9-9173-4763c76b398e,0.0,0.0,0.0,1.0
...,...,...,...,...
ffff6695-b64d-4a67-aa14-34b3b7f63c3f,0.0,0.0,0.0,2.0
ffff7d69-b706-4c64-9d6d-da57a04bc32b,0.0,0.0,0.0,1.0
ffff8044-2a22-4846-8a72-999e870abbe9,0.0,0.0,0.0,1.0


In [15]:
sesiones.mean()

evento
cart                1.288066
purchase            0.285983
remove_from_cart    0.919972
view                2.155699
dtype: float64

In [16]:
# Reordenamos el dataset con más sentido desde le punto de vista del proceso del funnel
sesiones = sesiones[['view','cart','remove_from_cart','purchase']]

In [17]:
sesiones.mean()

evento
view                2.155699
cart                1.288066
remove_from_cart    0.919972
purchase            0.285983
dtype: float64